## 1.0 Calculo do ROI médio para esse projeto

In [5]:
import pandas as pd
import warnings

data = pd.read_csv("../data/HR-Employee-Attrition-balanced.csv")
data.head().T

,0,1,2,3,4
Age,58,27,50,39,32
Attrition,Yes,Yes,Yes,No,No
BusinessTravel,Travel_Rarely,Travel_Frequently,Travel_Rarely,Non-Travel,Travel_Rarely
DailyRate,601,1341,407,439,601
Department,Research & Development,Human Resources,Sales,Research & Development,Sales
DistanceFromHome,7,21,28,9,7
Education,4,2,3,3,5
EducationField,Medical,Human Resources,Marketing,Life Sciences,Marketing
EmployeeCount,1,1,1,1,1
EmployeeNumber,1360,1950,2049,1132,1446


In [ ]:
# Colunas de interesse
# Attrition
# MonthlyIncome
# JobLevel

In [40]:
print("="*60)
# Numero total de trabalhadores
print(f"Numero total de trabalhadores: {len(data)}")

# Numero total de trabalhadores que saíram
total_left =data["Attrition"].value_counts()["Yes"]
print(f"Numero total de trabalhadores que saíram: {total_left}")

# Salário total anual dos trabalhadores
total_salary_total = (data["MonthlyIncome"].sum())*12
print(f"Salário total anual dos trabalhadores: {total_salary_total:.2f}")

# Salário total anual dos trabalhadores que sairam
total_salary_attrition = (data[data["Attrition"]=="Yes"]["MonthlyIncome"].sum())*12
print(f"Salário total anual dos trabalhadores que saíram: {total_salary_attrition:.2f}")

# Porcentagem dos salarios dos trabalhadores que sairam
percentual_salario = (total_salary_attrition/total_salary_total)
print(f"Porcentagem dos salarios dos trabalhadores que sairam: {percentual_salario:.2%}")

# Custo de turnover por funcionário - ver regra de negócio no PROBLEMA_NEGOCIO.MD

Numero total de trabalhadores: 1761
Numero total de trabalhadores que saíram: 528
Salário total anual dos trabalhadores: 132937080.00
Salário total anual dos trabalhadores que saíram: 31839864.00
Porcentagem dos salarios dos trabalhadores que sairam: 23.95%


In [ ]:
# Adicionando uma nova coluna de valor de perda por saida
data_attrition = data.copy()

data_attrition_select = data_attrition.loc[
    data_attrition["Attrition"] == "Yes",
    :  # exemplo de colunas
]

data_attrition_select.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,58,Yes,Travel_Rarely,601,Research & Development,7,4,Medical,1,1360,...,4,80,0,31,0,2,10,9,5,9
1,27,Yes,Travel_Frequently,1341,Human Resources,21,2,Human Resources,1,1950,...,1,80,0,0,2,3,0,0,0,0
2,50,Yes,Travel_Rarely,407,Sales,28,3,Marketing,1,2049,...,2,80,1,19,3,2,2,2,1,0
5,29,Yes,Travel_Frequently,337,Research & Development,14,1,Other,1,1421,...,1,80,0,9,1,3,8,7,7,7
7,30,Yes,Travel_Frequently,1441,Research & Development,0,5,Life Sciences,1,1442,...,1,80,0,10,1,2,10,7,3,7


In [52]:
# Criando uma regra para o joblevel
# Criando uma coluna de taxa de perda por nivel 
# Se JobLevel == 1 retorne 0.4
# Se JobLevel == 2 retorne 1
# Se JobLevel == 3 retorne 1.5
# Se JobLevel == 4 ou > retorne 2

data_attrition_select["taxa_perda_por_nivel"] = data_attrition_select["JobLevel"].apply(
    lambda x:
        0.4 if x == 1 else
        1 if x == 2 else
        1.5 if x == 3 else
        2
)
data_attrition_select.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,taxa_perda_por_nivel
0,58,Yes,Travel_Rarely,601,Research & Development,7,4,Medical,1,1360,...,80,0,31,0,2,10,9,5,9,1.5
1,27,Yes,Travel_Frequently,1341,Human Resources,21,2,Human Resources,1,1950,...,80,0,0,2,3,0,0,0,0,2.0
2,50,Yes,Travel_Rarely,407,Sales,28,3,Marketing,1,2049,...,80,1,19,3,2,2,2,1,0,1.0
5,29,Yes,Travel_Frequently,337,Research & Development,14,1,Other,1,1421,...,80,0,9,1,3,8,7,7,7,1.5
7,30,Yes,Travel_Frequently,1441,Research & Development,0,5,Life Sciences,1,1442,...,80,0,10,1,2,10,7,3,7,1.5


In [54]:
# Calculando a perda financeira para cada funcionário que sair

data_attrition_select["salario_anual"] = data_attrition_select["MonthlyIncome"]*12
data_attrition_select["perda_financeira"] = data_attrition_select["salario_anual"] * data_attrition_select["taxa_perda_por_nivel"]
data_attrition_select.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,taxa_perda_por_nivel,salario_anual,perda_financeira
0,58,Yes,Travel_Rarely,601,Research & Development,7,4,Medical,1,1360,...,31,0,2,10,9,5,9,1.5,120096,180144.0
1,27,Yes,Travel_Frequently,1341,Human Resources,21,2,Human Resources,1,1950,...,0,2,3,0,0,0,0,2.0,33564,67128.0
2,50,Yes,Travel_Rarely,407,Sales,28,3,Marketing,1,2049,...,19,3,2,2,2,1,0,1.0,131736,131736.0
5,29,Yes,Travel_Frequently,337,Research & Development,14,1,Other,1,1421,...,9,1,3,8,7,7,7,1.5,90636,135954.0
7,30,Yes,Travel_Frequently,1441,Research & Development,0,5,Life Sciences,1,1442,...,10,1,2,10,7,3,7,1.5,88860,133290.0


In [59]:
print("="*60)
# Numero total de trabalhadores
print(f"Numero total de trabalhadores: {len(data)}")

# Numero total de trabalhadores que saíram
total_left =data["Attrition"].value_counts()["Yes"]
print(f"Numero total de trabalhadores que saíram: {total_left}")

# Taxa de trabalhadores que sairam
taxa_saida = total_left/len(data)
print(f"Taxa de trabalhadores que sairam: {taxa_saida:.2%}")

# Salário total anual dos trabalhadores
total_salary_total = (data["MonthlyIncome"].sum())*12
print(f"Salário total anual dos trabalhadores: {total_salary_total:.2f}")

# Salário total anual dos trabalhadores que sairam
total_salary_attrition = (data[data["Attrition"]=="Yes"]["MonthlyIncome"].sum())*12
print(f"Salário total anual dos trabalhadores que saíram: {total_salary_attrition:.2f}")

# Porcentagem dos salarios dos trabalhadores que sairam
percentual_salario = (total_salary_attrition/total_salary_total)
print(f"Porcentagem dos salarios dos trabalhadores que sairam: {percentual_salario:.2%}")

# Custo de turnover por funcionário - ver regra de negócio no PROBLEMA_NEGOCIO.MD
custo_total_funcionario_perda = data_attrition_select["perda_financeira"].sum()
print(f"Custo total por turnover: {custo_total_funcionario_perda:.2f}")

# Custo médio de turnover por funcionário
custo_medio_turnover = data_attrition_select["perda_financeira"].mean()
print(f"Custo médio de turnover: {custo_medio_turnover:.2f}")

Numero total de trabalhadores: 1761
Numero total de trabalhadores que saíram: 528
Taxa de trabalhadores que sairam: 29.98%
Salário total anual dos trabalhadores: 132937080.00
Salário total anual dos trabalhadores que saíram: 31839864.00
Porcentagem dos salarios dos trabalhadores que sairam: 23.95%
Custo total por turnover: 37495244.40
Custo médio de turnover: 71013.72
